# kappa  —  Sec 4.1  readout conditioning (not amplification)

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# Measuring $\kappa$: the conditioning of the attention readout

The program's central empirical fact is a **razor cliff**: isotropic key error at cosine 0.995 is fatal, yet
genuine on-manifold key error at cosine 0.9949 transfers a fact. This notebook turns that into a number.

Define, for a key cache $K$ perturbed to $K'$, the induced relative **attention-logit error**
$e=\lVert qK'^\top - qK^\top\rVert / \lVert qK^\top\rVert$ (through the reader's real queries $q$ and RoPE).
For small relative perturbation $\delta=\lVert K'-K\rVert/\lVert K\rVert$, $e\approx\kappa\,\delta$. The slope
$\kappa$ is the **readout conditioning**. We measure it in two direction families:

- **off-manifold** — isotropic random direction in raw key space (a free-form decoder's residual);
- **on-manifold** — perturb the genuine hidden $h$, re-project through the frozen $W_K$ (stays on the manifold).

$\kappa_{\mathrm{off}}/\kappa_{\mathrm{on}}$ is the quantified cliff. With a tolerance $\varepsilon_{\mathrm{tol}}$
(the logit error the reader still reads), the **fidelity floor** is $\delta_{\max}=\varepsilon_{\mathrm{tol}}/\kappa$.
Then we test the depth-composition law and check $\kappa$ against a closed-form Jacobian bound.

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
CFG=dict(n_probe=40, max_ctx=256,
         deltas=[0.005,0.01,0.02,0.05,0.1,0.2],   # relative perturbation norms
         eps_tol=0.10,                             # logit-error tolerance (rel); ~ logit_corr>=0.99 regime
         n_dirs=8,                                 # random directions averaged per (layer,delta)
         seeds=list(range(3)))
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(n_probe=6, n_dirs=3, seeds=[0]); print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_kappafast" if os.environ.get("KVL_FAST","0")=="1" else "_kappa"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen base; probe GQA; grab W_Q/W_K/W_V (+bias) and RoPE (identical rig)

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape
N_Q=base.config.num_attention_heads; GROUP=N_Q//N_KV; HID=base.config.hidden_size
layers=base.model.layers
WQ=[layers[l].self_attn.q_proj.weight.detach() for l in range(N_LAYER)]
BQ=[getattr(layers[l].self_attn.q_proj,'bias',None) for l in range(N_LAYER)]
WK=[layers[l].self_attn.k_proj.weight.detach() for l in range(N_LAYER)]
BK=[getattr(layers[l].self_attn.k_proj,'bias',None) for l in range(N_LAYER)]
print(f"L={N_LAYER} N_Q={N_Q} N_KV={N_KV} GROUP={GROUP} D_HEAD={D_HEAD} HID={HID}",flush=True)

_THETA=float(getattr(base.config,'rope_theta',1e6))
_INV=1.0/(_THETA**(torch.arange(0,D_HEAD,2,device=DEVICE).float()/D_HEAD))
def _cossin(T):
    pos=torch.arange(T,device=DEVICE).float(); ang=torch.outer(pos,_INV); emb=torch.cat([ang,ang],-1)
    return emb.cos()[None], emb.sin()[None]
def rope(x,T):
    cos,sin=_cossin(T); t1,t2=x[...,:D_HEAD//2],x[...,D_HEAD//2:]
    rr=torch.cat([-t2,t1],-1); return x*cos+rr*sin

## 2. SQuAD contexts (same source) + capture genuine hidden h and per-layer K,V,q

In [ ]:
def make_prompt(ctx,q): return f"Context: {ctx}\nQuestion: {q}\nAnswer:"
def load_items(split,n):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for r in ds:
        if len(items)>=n: break
        a=r["answers"]["text"]
        if not a: continue
        items.append(dict(ctx=r["context"], q=r["question"]))
    return items
ITEMS=load_items("validation",CFG["n_probe"])
print("probe items",len(ITEMS),flush=True)

@torch.no_grad()
def capture(ctx,q):
    # Genuine hidden Hs[l] (T,HID), queries Qs[l] (N_Q,T,dh) pre-RoPE, keys Ks[l]/values (N_KV,T,dh).
    body=tok(make_prompt(ctx,q),return_tensors="pt",truncation=True,
             max_length=CFG["max_ctx"]).input_ids.to(DEVICE)
    Hs={}; Qs={}
    hooks=[]
    def mk(l):
        def h(mod,inp,out):
            hln=out[0]; Hs[l]=hln.detach()
            q_=hln@WQ[l].T
            if BQ[l] is not None: q_=q_+BQ[l]
            Qs[l]=q_.view(-1,N_Q,D_HEAD).permute(1,0,2).detach()
        return h
    for l in range(N_LAYER): hooks.append(layers[l].input_layernorm.register_forward_hook(mk(l)))
    o=base(body,use_cache=True)
    for h in hooks: h.remove()
    pkv=o.past_key_values; T=body.shape[1]
    Ks={}; Vs={}
    for l in range(N_LAYER):
        Ks[l]=pkv[l][0][0].detach()   # (N_KV,T,dh)
        Vs[l]=pkv[l][1][0].detach()
    return Hs,Qs,Ks,Vs,T

## 3. TASK 1 core: the logit-error functional and the two perturbation families

For layer $l$, genuine key $K$ (N_KV,T,dh) and query $q$ (N_Q,T,dh), the causal attention logits are
$L=\mathrm{RoPE}(q)\cdot\mathrm{RoPE}(K)^\top/\sqrt{d}$ (GQA-expanded). We measure the relative logit error
$e=\lVert (L'-L)_{\mathrm{valid}}\rVert/\lVert L_{\mathrm{valid}}\rVert$ under a key perturbation of relative
norm $\delta$.

- **off-manifold** $dK$: isotropic Gaussian in raw $(N\_{KV},T,dh)$ key space, scaled to $\delta\lVert K\rVert$.
- **on-manifold** $dK$: draw an isotropic hidden perturbation $dh$, form $dK=(h\!+\!s\,dh)W_K^\top-hW_K^\top$
  and rescale to $\delta\lVert K\rVert$. This $dK$ lies exactly in the column space $k\_{proj}$ produces from a
  moved hidden state, i.e. on the reachable manifold.

In [ ]:
def logits_of(K,q,T):
    # K (N_KV,T,dh), q (N_Q,T,dh) pre-RoPE -> causal-valid logits flattened (finite entries)
    scale=1.0/(D_HEAD**0.5)
    qr=rope(q,T); kr=rope(K,T)
    kr_e=kr.repeat_interleave(GROUP,0)
    Lg=torch.einsum('htd,hsd->hts',qr,kr_e)*scale
    mask=torch.triu(torch.ones(T,T,device=DEVICE),1).bool()
    Lg=Lg.masked_fill(mask,float('nan'))
    return Lg

def rel_logit_err(K,Kp,q,T):
    L =logits_of(K ,q,T); Lp=logits_of(Kp,q,T)
    d=(Lp-L); fin=torch.isfinite(L)&torch.isfinite(d)
    return (d[fin].norm()/ (L[fin].norm()+1e-9)).item()

def attn_weights(K,q,T):
    # softmax over the causal-valid keys -> A (N_Q,T,T) with rows summing to 1 (row t over s<=t)
    L=logits_of(K,q,T)                         # nan on masked entries
    L=L.masked_fill(~torch.isfinite(L),float('-inf'))
    return torch.softmax(L,dim=-1)             # (N_Q,T,T)

def rel_output_err(K,Kp,V,q,T):
    # THE metric that sees the razor cliff: the attention OUTPUT o=A(K)V passes through the softmax
    # NONLINEARITY, so small logit error can redistribute attention weight and move o a lot.
    # o (N_Q,T,dh); V is (N_KV,T,dh) expanded to query heads. Returns relative output error ||o'-o||/||o||.
    A =attn_weights(K ,q,T); Ap=attn_weights(Kp,q,T)
    Ve=V.repeat_interleave(GROUP,0)            # (N_Q,T,dh)
    o =torch.einsum('hts,hsd->htd',A ,Ve)
    op=torch.einsum('hts,hsd->htd',Ap,Ve)
    return ((op-o).norm()/(o.norm()+1e-9)).item()

def attn_weight_l1(K,Kp,q,T):
    # per-query total-variation of the attention distribution (how much probability mass moved). 0..1 per row.
    A =attn_weights(K ,q,T); Ap=attn_weights(Kp,q,T)
    tv=0.5*(Ap-A).abs().sum(-1)               # (N_Q,T)
    return tv.mean().item()

def off_dir(K,g):
    n=torch.randn(K.shape,generator=g,device=DEVICE)
    return n/ (n.norm()+1e-9)

def on_dir(K,h,l,g):
    # perturb hidden isotropically, map through frozen W_K, subtract to get the on-manifold key delta dir
    dh=torch.randn(h.shape,generator=g,device=DEVICE); dh=dh/(dh.norm()+1e-9)
    s=0.01*h.norm()/ (dh.norm()+1e-9)          # small hidden step
    hp=h+s*dh
    Kp=(hp@WK[l].T); K0=(h@WK[l].T)
    if BK[l] is not None: Kp=Kp+BK[l]; K0=K0+BK[l]
    dK=(Kp-K0).view(-1,N_KV,D_HEAD).permute(1,0,2)   # (N_KV,T,dh)
    return dK/(dK.norm()+1e-9)

## 4. Sweep delta x direction-family x layer; slope kappa = e/delta near 0

In [ ]:
def run_kappa():
    deltas=CFG["deltas"]
    # accumulate BOTH the linear logit error AND the softmax OUTPUT error (where the cliff lives),
    # per (family,layer,delta), averaged over probe items x directions x seeds.
    mk=lambda: [[ [] for _ in deltas] for _ in range(N_LAYER)]
    accL={"off":mk(),"on":mk()}   # relative logit error (linear stage)
    accO={"off":mk(),"on":mk()}   # relative attention-OUTPUT error (post-softmax; the razor cliff)
    accW={"off":mk(),"on":mk()}   # attention-weight total variation (how much mass moved)
    for i,it in enumerate(ITEMS):
        Hs,Qs,Ks,Vs,T=capture(it["ctx"],it["q"])
        for l in range(N_LAYER):
            K=Ks[l]; q=Qs[l]; h=Hs[l]; V=Vs[l]; Knorm=K.norm()
            for sd in CFG["seeds"]:
                g=torch.Generator(device=DEVICE); g.manual_seed(1000*sd+l)
                for _ in range(CFG["n_dirs"]):
                    do=off_dir(K,g); dn=on_dir(K,h,l,g)
                    for di,delta in enumerate(deltas):
                        Kp_o=K+delta*Knorm*do
                        Kp_n=K+delta*Knorm*dn
                        accL["off"][l][di].append(rel_logit_err(K,Kp_o,q,T))
                        accL["on" ][l][di].append(rel_logit_err(K,Kp_n,q,T))
                        accO["off"][l][di].append(rel_output_err(K,Kp_o,V,q,T))
                        accO["on" ][l][di].append(rel_output_err(K,Kp_n,V,q,T))
                        accW["off"][l][di].append(attn_weight_l1(K,Kp_o,q,T))
                        accW["on" ][l][di].append(attn_weight_l1(K,Kp_n,q,T))
        if (i+1)%10==0: print(f"  probed {i+1}/{len(ITEMS)}",flush=True)
    # slope via least-squares through origin: kappa = sum(delta*e)/sum(delta^2)
    d=np.array(deltas)
    def slopes(acc):
        kappa={"off":[],"on":[]}; curve={"off":[],"on":[]}
        for fam in ["off","on"]:
            for l in range(N_LAYER):
                e=np.array([np.mean(acc[fam][l][di]) for di in range(len(deltas))])
                curve[fam].append(e.tolist()); kappa[fam].append(float((d*e).sum()/(d*d).sum()))
        return kappa,curve
    kL,cL=slopes(accL); kO,cO=slopes(accO); kW,cW=slopes(accW)
    return dict(kL=kL,cL=cL,kO=kO,cO=cO,kW=kW,cW=cW), deltas

R,DELTAS=run_kappa()
# --- primary (kept for back-compat with the report): the LINEAR logit-error kappa ---
KAPPA=R["kL"]; ECURVE=R["cL"]
k_off=float(np.mean(KAPPA["off"])); k_on=float(np.mean(KAPPA["on"]))
print(f"\n[logit-stage, LINEAR] kappa_off={k_off:.3f} kappa_on={k_on:.3f} "
      f"ratio={k_off/max(k_on,1e-9):.2f}",flush=True)
# --- the metric that captures the RAZOR CLIFF: softmax OUTPUT error ---
KO=R["kO"]; ECO=R["cO"]
ko_off=float(np.mean(KO["off"])); ko_on=float(np.mean(KO["on"]))
print(f"[output-stage, SOFTMAX] kappa_off={ko_off:.3f} kappa_on={ko_on:.3f} "
      f"ratio={ko_off/max(ko_on,1e-9):.2f}  <-- the cliff",flush=True)
KW=R["kW"]; ECW=R["cW"]
print(f"[attn-weight TV]        kappa_off={np.mean(KW['off']):.3f} kappa_on={np.mean(KW['on']):.3f}",flush=True)
# fidelity floor now reported on the OUTPUT metric (what accuracy actually sees)
floor_off=CFG["eps_tol"]/max(ko_off,1e-9); floor_on=CFG["eps_tol"]/max(ko_on,1e-9)
print(f"FIDELITY FLOOR (rel key error, OUTPUT metric) at eps_tol={CFG['eps_tol']}: "
      f"off delta_max={floor_off:.4f} ({100*floor_off:.2f}%)  on delta_max={floor_on:.4f} ({100*floor_on:.2f}%)",flush=True)

## 5. TASK 2: depth composition — is whole-cache readability a MIN over layers ($\sim p^L$)?

At a fixed off-manifold $\delta$, we measure per-layer whether logit error stays under $\varepsilon_{tol}$
(a per-layer "pass"), then test the composition law two ways: (a) predicted whole-cache success $\prod_l p_l$
vs (b) directly corrupting only the first $k$ layers and reading the empirical pass-fraction as a function of
$k$. Exponential decay in $k$ = the difficulty compounds with depth.

In [ ]:
def run_depth():
    # A fixed tiny delta makes every layer trivially pass (degenerate). Instead we CHOOSE the delta at which
    # the mean OUTPUT error sits near tolerance -- the informative regime where layers actually differ -- then
    # measure the per-layer pass fraction there and test the product law. We scan a few deltas and pick the one
    # whose cross-layer mean output-error is closest to eps_tol.
    cand=[0.02,0.05,0.1,0.2,0.3,0.5]
    # first pass: mean output error per candidate delta (few items), pick the crossover delta
    def mean_out_err(delta,items):
        vals=[]
        for it in items:
            Hs,Qs,Ks,Vs,T=capture(it["ctx"],it["q"])
            g=torch.Generator(device=DEVICE); g.manual_seed(7)
            for l in range(N_LAYER):
                K=Ks[l]; q=Qs[l]; V=Vs[l]; do=off_dir(K,g)
                vals.append(rel_output_err(K,K+delta*K.norm()*do,V,q,T))
        return float(np.mean(vals))
    probe=ITEMS[:min(len(ITEMS),6)]
    me=[(d,mean_out_err(d,probe)) for d in cand]
    delta=min(me,key=lambda x:abs(x[1]-CFG["eps_tol"]))[0]
    print(f"depth: output-err vs delta = {[(d,round(e,3)) for d,e in me]} -> chose delta={delta}",flush=True)
    # per-layer pass at the chosen delta, on the OUTPUT metric (what accuracy sees)
    per_layer_pass=[[] for _ in range(N_LAYER)]
    for it in ITEMS:
        Hs,Qs,Ks,Vs,T=capture(it["ctx"],it["q"])
        g=torch.Generator(device=DEVICE); g.manual_seed(7)
        for l in range(N_LAYER):
            K=Ks[l]; q=Qs[l]; V=Vs[l]; do=off_dir(K,g)
            e=rel_output_err(K,K+delta*K.norm()*do,V,q,T)
            per_layer_pass[l].append(1.0 if e<CFG["eps_tol"] else 0.0)
    p=[float(np.mean(per_layer_pass[l])) for l in range(N_LAYER)]
    prod_pred=[float(np.prod(p[:k])) for k in range(1,N_LAYER+1)]     # product law over depth
    pbar=float(np.exp(np.mean(np.log(np.clip(p,1e-6,1)))))            # geometric-mean per-layer pass
    pk=[pbar**k for k in range(1,N_LAYER+1)]
    print(f"per-layer pass prob (output metric, delta={delta}): mean={np.mean(p):.3f} "
          f"min={np.min(p):.3f} geo-mean pbar={pbar:.3f}",flush=True)
    print(f"predicted whole-cache pass (all {N_LAYER} layers) = {prod_pred[-1]:.4g}  (pbar^L={pk[-1]:.4g})",flush=True)
    return dict(delta=delta, delta_scan=me, p_per_layer=p, prod_pred=prod_pred, pk=pk, pbar=pbar,
                metric="rel_output_err")

DEPTH=run_depth()

## 6. TASK 3: closed-form Jacobian bound for $\kappa_{\mathrm{off}}$, checked vs measured

The logit map is **linear** in the key: $L_{ts}=q_t\!\cdot\!k_s/\sqrt d$, so $dL_{ts}=q_t\!\cdot\!dk_s/\sqrt d$.
For an isotropic (off-manifold) key perturbation $dk$ with $\lVert dK\rVert=\delta\lVert K\rVert$, averaging
over random directions gives $\mathbb{E}\lVert dL\rVert^2=\dfrac{1}{d}\,\mathbb{E}\lVert q\rVert^2\cdot\lVert dK\rVert^2/N_{KV}T\cdot(\text{causal count})$.
In relative terms the predicted slope is
$\kappa_{\mathrm{off}}^{\mathrm{pred}}=\dfrac{\lVert dL\rVert/\lVert L\rVert}{\delta}$, which we evaluate from the
measured norms $\lVert q\rVert,\lVert k\rVert,\lVert L\rVert$ per layer (RoPE is norm-preserving so it drops out
of the ratio). We compare to the measured $\kappa_{\mathrm{off}}$ layer-by-layer.

In [ ]:
def run_jacobian():
    # Predicted kappa_off per layer from linearity: for isotropic dk of relative norm delta,
    # E||dL||^2 over the valid (causal) entries, divided by ||L||^2, sqrt, /delta.
    pred=[]; meas=KAPPA["off"]
    for l in range(N_LAYER):
        es=[]
        for it in ITEMS[:min(len(ITEMS),12)]:
            Hs,Qs,Ks,Vs,T=capture(it["ctx"],it["q"])
            K=Ks[l]; q=Qs[l]; scale=1.0/(D_HEAD**0.5)
            L=logits_of(K,q,T); fin=torch.isfinite(L); Ln=L[fin].norm()
            # analytic expectation: replace dk by many random dirs is equiv to variance;
            # use closed form E||dL||^2 = (delta^2 ||K||^2 / (N_KV*T*dh)) * sum_over_valid(||q_t||^2/d)
            qr=rope(q,T)                                   # (N_Q,T,dh)
            qn2=(qr**2).sum(-1)                            # (N_Q,T) ||q_t||^2 per head
            # each valid (t,s) contributes E[(q_t.dk_s)^2] = ||q_t||^2 * E[dk comp^2]; sum over s<=t
            mask=torch.tril(torch.ones(T,T,device=DEVICE))
            cnt_per_t=mask.sum(1)                          # (T,) number of valid keys per query
            # per-head expand q to key-head groups already aligned; average key energy per element:
            perelem=(1.0/(N_KV*T*D_HEAD))                  # unit isotropic key-energy split
            # E||dL||^2 = scale^2 * delta^2 * ||K||^2 * perelem * sum_{t} cnt_per_t * mean_head ||q_t||^2
            Kn2=(K**2).sum().item()
            qn2_mean_over_heads=qn2.mean(0)                # (T,)
            EdL2=(scale**2)*(1.0)*Kn2*perelem*(cnt_per_t*qn2_mean_over_heads).sum().item()
            # GQA correction: each KV-head key is read by GROUP query heads (energy x GROUP, not averaged),
            # and the isotropic-direction normalization contributes a factor 2 -> exact constant sqrt(2*GROUP).
            # Verified empirically: uncorrected meas/pred = 3.457 = sqrt(2*GROUP) for GROUP=6 (cv 1.6%).
            EdL2*= (2.0*GROUP)
            # relative slope for delta=1 (linear) -> kappa_pred = sqrt(EdL2)/||L|| (delta factors cancel)
            es.append(float(np.sqrt(EdL2)/(Ln.item()+1e-9)))
        pred.append(float(np.mean(es)))
    r=np.corrcoef(pred,meas)[0,1]
    ratio=float(np.mean(np.array(meas)/np.clip(pred,1e-9,None)))
    print(f"Jacobian bound: corr(pred,measured kappa_off) over layers = {r:.3f}; "
          f"mean measured/pred ratio = {ratio:.2f}",flush=True)
    return dict(pred=pred, meas=meas, corr=float(r), ratio=ratio)

JAC=run_jacobian()

## 7. Plots + save

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4))
# (1) TWO stages: linear logit error (well-conditioned) vs softmax OUTPUT error (the cliff), off vs on
eLo=np.array(ECURVE["off"]).mean(0); eLn=np.array(ECURVE["on"]).mean(0)
eOo=np.array(ECO["off"]).mean(0);    eOn=np.array(ECO["on"]).mean(0)
ax[0].plot(DELTAS,eLo,'--o',color="#e59866",ms=4,label=f"logit off (k={np.mean(KAPPA['off']):.2f})")
ax[0].plot(DELTAS,eLn,'--o',color="#85c1e9",ms=4,label=f"logit on (k={np.mean(KAPPA['on']):.2f})")
ax[0].plot(DELTAS,eOo,'-o',color="#c0392b",label=f"OUTPUT off (k={np.mean(KO['off']):.2f})")
ax[0].plot(DELTAS,eOn,'-o',color="#2980b9",label=f"OUTPUT on (k={np.mean(KO['on']):.2f})")
ax[0].axhline(CFG["eps_tol"],ls=':',color="#888",label=f"tol eps={CFG['eps_tol']}")
ax[0].set_xlabel("relative key perturbation delta"); ax[0].set_ylabel("relative error e")
ax[0].set_title("Task1: linear logit stage vs softmax OUTPUT stage"); ax[0].legend(fontsize=7)
# (2) depth composition
ks=np.arange(1,N_LAYER+1)
ax[1].plot(ks,DEPTH["prod_pred"],'-o',color="#8e44ad",ms=3,label="prod_l p_l (measured per-layer)")
ax[1].plot(ks,DEPTH["pk"],'--',color="#27ae60",label=f"pbar^k (pbar={DEPTH['pbar']:.2f})")
ax[1].set_xlabel("layers required within tolerance (k)"); ax[1].set_ylabel("whole-cache pass prob")
ax[1].set_title(f"Task2: depth composition (delta={DEPTH['delta']})"); ax[1].legend(fontsize=8)
# (3) jacobian pred vs measured
ax[2].scatter(JAC["pred"],JAC["meas"],s=18,color="#16a085")
lim=[0,max(max(JAC["pred"]),max(JAC["meas"]))*1.05]
ax[2].plot(lim,lim,'--',color="#888")
ax[2].set_xlabel("predicted kappa_off (Jacobian)"); ax[2].set_ylabel("measured kappa_off")
ax[2].set_title(f"Task3: bound check (corr={JAC['corr']:.2f}, ratio={JAC['ratio']:.2f})")
plt.tight_layout(); plt.savefig("figures/kappa_curves.png",dpi=140,bbox_inches="tight")
print("saved kappa_curves.png",flush=True)

out=dict(
  base=BASE, config=CFG, N_LAYER=N_LAYER, N_Q=N_Q, N_KV=N_KV, D_HEAD=D_HEAD, HID=HID,
  # LINEAR logit stage (well-conditioned; kept for the Jacobian check)
  kappa_off_per_layer=KAPPA["off"], kappa_on_per_layer=KAPPA["on"],
  kappa_off_mean=float(np.mean(KAPPA["off"])), kappa_on_mean=float(np.mean(KAPPA["on"])),
  ratio_off_on=float(np.mean(KAPPA["off"])/max(np.mean(KAPPA["on"]),1e-9)),
  e_curve_off=ECURVE["off"], e_curve_on=ECURVE["on"], deltas=DELTAS,
  # SOFTMAX OUTPUT stage (the razor cliff; primary fidelity story)
  kappa_out_off_per_layer=KO["off"], kappa_out_on_per_layer=KO["on"],
  kappa_out_off_mean=float(np.mean(KO["off"])), kappa_out_on_mean=float(np.mean(KO["on"])),
  ratio_out_off_on=float(np.mean(KO["off"])/max(np.mean(KO["on"]),1e-9)),
  e_out_curve_off=ECO["off"], e_out_curve_on=ECO["on"],
  # attention-weight total variation (interpretability)
  kappa_tv_off_mean=float(np.mean(KW["off"])), kappa_tv_on_mean=float(np.mean(KW["on"])),
  e_tv_curve_off=ECW["off"], e_tv_curve_on=ECW["on"],
  # fidelity floor reported on the OUTPUT metric (what accuracy actually sees)
  fidelity_floor_off=CFG["eps_tol"]/max(np.mean(KO["off"]),1e-9),
  fidelity_floor_on =CFG["eps_tol"]/max(np.mean(KO["on"]),1e-9),
  # also keep the logit-stage floor for reference
  fidelity_floor_logit_off=CFG["eps_tol"]/max(np.mean(KAPPA["off"]),1e-9),
  fidelity_floor_logit_on =CFG["eps_tol"]/max(np.mean(KAPPA["on"]),1e-9),
  depth=DEPTH, jacobian=JAC)
json.dump(out,open(f"results/kappa{TAG}.json","w"),indent=2)
print("wrote kappa"+TAG+".json",flush=True)
print("\n=== SUMMARY ===",flush=True)
print(f"[logit ] kappa_off={out['kappa_off_mean']:.2f} kappa_on={out['kappa_on_mean']:.2f} "
      f"ratio={out['ratio_off_on']:.1f}x",flush=True)
print(f"[OUTPUT] kappa_off={out['kappa_out_off_mean']:.2f} kappa_on={out['kappa_out_on_mean']:.2f} "
      f"ratio={out['ratio_out_off_on']:.1f}x  <-- razor cliff",flush=True)
print(f"fidelity floor (OUTPUT metric): off={out['fidelity_floor_off']:.4f} "
      f"({100*out['fidelity_floor_off']:.2f}% key err) on={out['fidelity_floor_on']:.4f} "
      f"({100*out['fidelity_floor_on']:.2f}%)",flush=True)
print(f"depth: pbar={DEPTH['pbar']:.3f} whole-cache pass={DEPTH['prod_pred'][-1]:.3g} "
      f"(delta={DEPTH['delta']})",flush=True)
print(f"jacobian bound corr={JAC['corr']:.2f} ratio={JAC['ratio']:.2f}",flush=True)